<table border=0 width="100%"><tr><td><p align="left"><img src="..\img\logo-1.jpg" align="left" width=300></p></td><td><font size=3><B>Data analysis (Python) - Tokenization (Jun Wang)</B></font></td></tr></table>

# content
* Tokenization
* Tokenization Chinese using jieba
* Cut sentences
* open

# Tokenization 

为了表示评论，需要界定评论中使用的词。例如“欢迎新老师生前来就餐”句子可能存在不同的切词结果，我们需要选择切词最好的结果。什么是最好呢？这里有些原则和优化目标。目前常见的方法有：
* 基于规则的方法（例如最大匹配方法，Maximum Match Method）
* 基于统计的方法（语言模型）

例如，根据词典{南京、南京市、市长、江、大桥、长江、长江大桥}对“南京市长江大桥”进行分词，切词结果是{南京市、长江大桥}

最大匹配方法的思路是：
* 从左向右取待切分汉语句的m个字符作为匹配字段，m为机器词典中最长词条的字符数。
* 查找词典并进行匹配。**若匹配成功，**则将这个匹配字段作为一个词切分出来。**若匹配不成功**，则将这个匹配字段的最后一个字去掉，剩下的字符串作为新的匹配字段，进行再次匹配，重复以上过程，直到切分出所有词为止。

请尝试使用上述规则切词“南京市长江大桥”

统计语言模型是单词序列的概率分布。对长度为$m$的序列,它分配一个概率$𝑝(𝑤_1,𝑤_2,…,𝑤_𝑛)$。我们选择联合概率较大的词序切词结果。

例如，**[欢迎,新老,师生,前来,就餐]**比**[欢迎,新老师,生前,来,就餐]**出现的概率大，因此，我们倾向于选择前者。

有两种方法可以计算一个句子出现的概率
* 要计算整个序列出现的概率，可以基于词语在预料中的词频及共现次数来计算条件概率，因此语言模型属于**基于统计的方法**。
* 当然，目前也可以**基于序列模型（例如循环神经网络）**来做切词。

我们这里用一个英语句子做例子，计算条件概率与句子出现的概率。

$$ P(the|its，water，is，so，transparent，that) = \frac{count(its，water，is，so，transparent，that，the)}{count(its，water，is，so，transparent，that)}$$

with a large enough corpus, such as the web, we can compute these counts and estimate the probability of a sentence as follows
$$P(w_1,w_2,...,w_n) = P(w_1)P(w_2|w_1)P(w_3|w_1,w_2)...P(w_n|w_1,...w_{n-1})$$

instead of computing the probability of a word given its **entire history**, we can approximate the history by just the last few words.

* n-gram model
$$P(ω_i|ω_1，ω_2，…，ω_{i-1})≈P(ωi|ω_{i-(n-1)}，…，ω{i-1})$$
* n = 1时为unigram model（聚焦一个词，也就是这个词本身）
$$P(ω_1，ω_2，…，ω_m)=P(ω_1)P(ω_2)…P(ω_m)$$
* n = 2时为bigram model，A Markov assumption（聚焦两个词）
$$P(ω_i|ω_1，ω_2，…，ω_{i-1})=P(ω_i|ω_{i-1})$$

一般使用频率计数的比例来计算n元条件概率
$$𝑃(𝑤_𝑛│𝑤_{𝑛−1})=  \frac{𝑐𝑜𝑢𝑛𝑡(𝑤_𝑛,𝑤_{𝑛−1})}{𝑐𝑜𝑢𝑛𝑡(𝑤_{𝑛−1})}$$

<img src="../img/n-gram.png" align='left' height="800" width="800"/>

# Tokenization Chinese using jieba

if you want to tokenize English, please try **nltk**. we do not cover this topic in class,you can learn it by yourself.

Install nltk

Natural Language Toolkit
provides easy-to-use interfaces to over 50 corpora and lexical resources such as WordNet
a suite of text processing libraries for classification, tokenization, stemming, tagging, parsing, and semantic reasoning, wrappers for industrial-strength NLP libraries

Install nltk - pip install nltk

In jupyter 
nltk.download()

http://www.nltk.org/ 

## use jieba to cut Chinese sentences

我们采用jieba工具包股评进行分词，更多资料参见[jieba](https://github.com/fxsjy/jieba)。jieba支持三种分词模式：
* 精确模式，试图将句子最精确地切开，适合文本分析；**默认模式**是精确模式
* 全模式，把句子中所有的可以成词的词语都扫描出来, 速度非常快，但是不能解决歧义；
* 搜索引擎模式，在精确模式的基础上，对长词再次切分，提高召回率，适合用于搜索引擎分词。

<img src="../img/1665567040607.jpg" align='left' height="800" width="800"/>

In [2]:
import jieba

In [3]:
# 我们选择一个评论来做切词演示
comment_text = "主板近期还是跟走钢丝一样，在高位箱体持续支撑，今天北上资金沪股通净流入30多亿，两市净流入43亿，北上资金有在护盘，还是有机会继续高位支撑一些。指数还得看创业板为主，近期筑底形态还在。看他后面洗盘后继续上攻，支撑位在20日均线到30日均线之间，主要是大科技回踩之后，等调仓低位新能源蹊跷版拉回去。明日看震荡，基本和今日一样，没波动的，下去还会再拉上来，总体还是震荡。"
comment_text

'主板近期还是跟走钢丝一样，在高位箱体持续支撑，今天北上资金沪股通净流入30多亿，两市净流入43亿，北上资金有在护盘，还是有机会继续高位支撑一些。指数还得看创业板为主，近期筑底形态还在。看他后面洗盘后继续上攻，支撑位在20日均线到30日均线之间，主要是大科技回踩之后，等调仓低位新能源蹊跷版拉回去。明日看震荡，基本和今日一样，没波动的，下去还会再拉上来，总体还是震荡。'

In [4]:
r = jieba.cut(comment_text)
print(r)
# 切词后的列表重新使用/连接成一个字符串
" ".join(r)

Building prefix dict from the default dictionary ...


<generator object Tokenizer.cut at 0x7f84b870cc10>


Dumping model to file cache /var/folders/kb/tv69ch3n1936k84ysf8skrph0000gn/T/jieba.cache
Loading model cost 0.525 seconds.
Prefix dict has been built successfully.


'主板 近期 还是 跟 走钢丝 一样 ， 在 高位 箱体 持续 支撑 ， 今天 北上 资金 沪 股通 净流入 30 多亿 ， 两市 净流入 43 亿 ， 北上 资金 有 在 护盘 ， 还是 有 机会 继续 高位 支撑 一些 。 指数 还 得 看 创业板 为主 ， 近期 筑底 形态 还 在 。 看 他 后面 洗盘 后 继续 上攻 ， 支撑位 在 20 日 均线 到 30 日 均线 之间 ， 主要 是 大 科技 回 踩 之后 ， 等 调仓 低位 新能源 蹊跷 版 拉回去 。 明日 看 震荡 ， 基本 和 今日 一样 ， 没 波动 的 ， 下去 还会 再 拉上来 ， 总体 还是 震荡 。'

用jieba.cut分割语句，得到一个generator object

In [5]:
# lcut切词结果保存在列表
r = jieba.lcut(comment_text)## lcut
print(r)
# 切词后的列表重新使用/连接成一个字符串
"/".join(r)

['主板', '近期', '还是', '跟', '走钢丝', '一样', '，', '在', '高位', '箱体', '持续', '支撑', '，', '今天', '北上', '资金', '沪', '股通', '净流入', '30', '多亿', '，', '两市', '净流入', '43', '亿', '，', '北上', '资金', '有', '在', '护盘', '，', '还是', '有', '机会', '继续', '高位', '支撑', '一些', '。', '指数', '还', '得', '看', '创业板', '为主', '，', '近期', '筑底', '形态', '还', '在', '。', '看', '他', '后面', '洗盘', '后', '继续', '上攻', '，', '支撑位', '在', '20', '日', '均线', '到', '30', '日', '均线', '之间', '，', '主要', '是', '大', '科技', '回', '踩', '之后', '，', '等', '调仓', '低位', '新能源', '蹊跷', '版', '拉回去', '。', '明日', '看', '震荡', '，', '基本', '和', '今日', '一样', '，', '没', '波动', '的', '，', '下去', '还会', '再', '拉上来', '，', '总体', '还是', '震荡', '。']


'主板/近期/还是/跟/走钢丝/一样/，/在/高位/箱体/持续/支撑/，/今天/北上/资金/沪/股通/净流入/30/多亿/，/两市/净流入/43/亿/，/北上/资金/有/在/护盘/，/还是/有/机会/继续/高位/支撑/一些/。/指数/还/得/看/创业板/为主/，/近期/筑底/形态/还/在/。/看/他/后面/洗盘/后/继续/上攻/，/支撑位/在/20/日/均线/到/30/日/均线/之间/，/主要/是/大/科技/回/踩/之后/，/等/调仓/低位/新能源/蹊跷/版/拉回去/。/明日/看/震荡/，/基本/和/今日/一样/，/没/波动/的/，/下去/还会/再/拉上来/，/总体/还是/震荡/。'

In [6]:

r = jieba.lcut(comment_text,cut_all=True)## 全模式
print(r)
# 切词后的列表重新使用/连接成一个字符串
"/".join(r)

['主板', '近期', '还是', '跟', '走钢丝', '钢丝', '一样', '，', '在', '高位', '箱体', '持续', '支撑', '，', '今天', '北上', '资金', '沪', '股', '通', '净流入', '流入', '30', '多亿', '，', '两', '市', '净流入', '流入', '43', '亿', '，', '北上', '资金', '有', '在', '护盘', '，', '还是', '有机', '机会', '继续', '高位', '支撑', '一些', '。', '指数', '还', '得', '看', '创业', '创业板', '为主', '，', '近期', '筑', '底', '形态', '还', '在', '。', '看', '他', '后面', '洗盘', '后继', '继续', '上攻', '，', '支撑', '支撑位', '在', '20', '日均', '均线', '到', '30', '日均', '均线', '之间', '，', '主要', '要是', '大', '科技', '回', '踩', '之后', '，', '等', '调', '仓', '低位', '新能源', '能源', '蹊跷', '版', '拉回', '拉回去', '回去', '。', '明日', '看', '震荡', '，', '基本', '和', '今日', '一样', '，', '没', '波动', '的', '，', '下去', '还', '会', '再', '拉上来', '上来', '，', '总体', '还是', '震荡', '。']


'主板/近期/还是/跟/走钢丝/钢丝/一样/，/在/高位/箱体/持续/支撑/，/今天/北上/资金/沪/股/通/净流入/流入/30/多亿/，/两/市/净流入/流入/43/亿/，/北上/资金/有/在/护盘/，/还是/有机/机会/继续/高位/支撑/一些/。/指数/还/得/看/创业/创业板/为主/，/近期/筑/底/形态/还/在/。/看/他/后面/洗盘/后继/继续/上攻/，/支撑/支撑位/在/20/日均/均线/到/30/日均/均线/之间/，/主要/要是/大/科技/回/踩/之后/，/等/调/仓/低位/新能源/能源/蹊跷/版/拉回/拉回去/回去/。/明日/看/震荡/，/基本/和/今日/一样/，/没/波动/的/，/下去/还/会/再/拉上来/上来/，/总体/还是/震荡/。'

In [7]:

r = jieba.cut_for_search(comment_text)## 搜索引擎模式
print(r)
# 切词后的列表重新使用/连接成一个字符串
"/".join(r)

<generator object Tokenizer.cut_for_search at 0x7f84b870ccf0>


'主板/近期/还是/跟/钢丝/走钢丝/一样/，/在/高位/箱体/持续/支撑/，/今天/北上/资金/沪/股通/流入/净流入/30/多亿/，/两市/流入/净流入/43/亿/，/北上/资金/有/在/护盘/，/还是/有/机会/继续/高位/支撑/一些/。/指数/还/得/看/创业/创业板/为主/，/近期/筑底/形态/还/在/。/看/他/后面/洗盘/后/继续/上攻/，/支撑/支撑位/在/20/日/均线/到/30/日/均线/之间/，/主要/是/大/科技/回/踩/之后/，/等/调仓/低位/能源/新能源/蹊跷/版/拉回/回去/拉回去/。/明日/看/震荡/，/基本/和/今日/一样/，/没/波动/的/，/下去/还会/再/上来/拉上来/，/总体/还是/震荡/。'

<font color=blue face=雅黑>思考：上面的切词结果有哪些您认为不合理的？</font>

## tokenization with user-defined dictionary

- 开发者可以[自定义词典](https://github.com/fxsjy/jieba)，以便包含 jieba 词库里没有的词。虽然 jieba 有新词识别能力，但是自行添加新词可以保证更高的正确率。
- 用法： jieba.load_userdict(file_name) # file_name 为文件类对象或自定义词典的路径。
- 词典格式和 dict.txt 一样，一个词占一行；每一行分三部分：词语、词频（可省略）、词性（可省略），用空格隔开，顺序不可颠倒。file_name 若为路径或二进制方式打开的文件，则文件必须为 UTF-8 编码。

**自定义词的例子**：巨亏，盘整，跑输，破发，深套，光线传媒

由于我们担心金融相关的词汇无法正确切出来，我们引入了三方面词汇，这三方面词汇我们统一保存在userdict.txt中。
* 金融情感词典（包括正面与负面情感词汇），来作为金融相关的用户自定义词典，该词典是西南财经大学教师李庆团队的[成果](https://www.sciencedirect.com/science/article/pii/S0167923614000232)；
* 同时，我们也引入了[王铁军](http://www.tiejun.wang/)整理的金融行业词汇，也把“光线传媒”也加入进来。
* 最后，我们通过人工阅读东方财富股吧评论文本，整理出一部分情感词汇。

In [8]:
comment = "我今天简直是巨亏，严重跑输大盘，破发深套其中，光线传媒啊，我该说点什么呢。"
r = jieba.lcut(comment) 
"/".join(r)

'我/今天/简直/是/巨亏/，/严重/跑/输/大盘/，/破发/深套/其中/，/光线/传媒/啊/，/我/该/说/点/什么/呢/。'

In [9]:
# 导入用户词典
jieba.load_userdict("userdict.txt")

In [10]:
r = jieba.lcut(comment)
"/".join(r)

'我/今天/简直/是/巨亏/，/严重/跑输/大盘/，/破发/深套/其中/，/光线传媒/啊/，/我/该/说/点/什么/呢/。'

# Cut sentences
* tips: try the split function

In [12]:
paragraph = """西南财经大学是教育部直属的国家“211工程”和“985工程”
优势学科创新平台建设的全国重点大学，
也是国家“双一流”建设高校。学校地处国家历史文化名城、南方丝绸之路的起点、素有“天府之国”美誉的成都，
有光华、柳林两校区，总占地2300余亩。光华铁树穿越年轮时光感受历史缅怀先贤,巍巍钟楼傲然屹立感受青春与未来对话,这里古今融通、
传统与现代交相辉映，
实乃读书治学的理想园地。学校始于1925年在上海创立的光华大学。
1938年，因抗战西迁建立光华大学成都分部；
1952-1953年汇聚西南地区17所院校的财经系科组建四川财经学院,这是建国初期国家按大区布局的四所本科财经院校之一，
也是西南地区唯一的综合性高等财经学府;1960年后历经分设、合并、更名等，于1978年恢复为四川财经学院；
1979年由四川省划归中国人民银行主管，逐渐形成了学校独特的金融行业背景和出色的金融学科优势；
1985年更名为西南财经大学，1997年成为国家“211工程”重点建设高校，2000年以独立建制划转教育部管理，
2011年成为国家“985工程”优势学科创新平台建设高校。90余年的办学历史形成了自己独特的办学特色、文化传统、精神内涵和社会声誉。
我们的西财是最可爱的校园!您认为呢？
"""
paragraph.split()

['西南财经大学是教育部直属的国家“211工程”和“985工程”',
 '优势学科创新平台建设的全国重点大学，',
 '也是国家“双一流”建设高校。学校地处国家历史文化名城、南方丝绸之路的起点、素有“天府之国”美誉的成都，',
 '有光华、柳林两校区，总占地2300余亩。光华铁树穿越年轮时光感受历史缅怀先贤,巍巍钟楼傲然屹立感受青春与未来对话,这里古今融通、',
 '传统与现代交相辉映，',
 '实乃读书治学的理想园地。学校始于1925年在上海创立的光华大学。',
 '1938年，因抗战西迁建立光华大学成都分部；',
 '1952-1953年汇聚西南地区17所院校的财经系科组建四川财经学院,这是建国初期国家按大区布局的四所本科财经院校之一，',
 '也是西南地区唯一的综合性高等财经学府;1960年后历经分设、合并、更名等，于1978年恢复为四川财经学院；',
 '1979年由四川省划归中国人民银行主管，逐渐形成了学校独特的金融行业背景和出色的金融学科优势；',
 '1985年更名为西南财经大学，1997年成为国家“211工程”重点建设高校，2000年以独立建制划转教育部管理，',
 '2011年成为国家“985工程”优势学科创新平台建设高校。90余年的办学历史形成了自己独特的办学特色、文化传统、精神内涵和社会声誉。',
 '我们的西财是最可爱的校园!您认为呢？']

下面代码参考了[中文分句re.split()](https://blog.csdn.net/zhuzuwei/article/details/80487032)

In [13]:
import re
sens = re.split('(。|\!|\.|\？)',paragraph) # try remove ()
sens

['西南财经大学是教育部直属的国家“211工程”和“985工程”\n优势学科创新平台建设的全国重点大学，\n也是国家“双一流”建设高校',
 '。',
 '学校地处国家历史文化名城、南方丝绸之路的起点、素有“天府之国”美誉的成都，\n有光华、柳林两校区，总占地2300余亩',
 '。',
 '光华铁树穿越年轮时光感受历史缅怀先贤,巍巍钟楼傲然屹立感受青春与未来对话,这里古今融通、\n传统与现代交相辉映，\n实乃读书治学的理想园地',
 '。',
 '学校始于1925年在上海创立的光华大学',
 '。',
 '\n1938年，因抗战西迁建立光华大学成都分部；\n1952-1953年汇聚西南地区17所院校的财经系科组建四川财经学院,这是建国初期国家按大区布局的四所本科财经院校之一，\n也是西南地区唯一的综合性高等财经学府;1960年后历经分设、合并、更名等，于1978年恢复为四川财经学院；\n1979年由四川省划归中国人民银行主管，逐渐形成了学校独特的金融行业背景和出色的金融学科优势；\n1985年更名为西南财经大学，1997年成为国家“211工程”重点建设高校，2000年以独立建制划转教育部管理，\n2011年成为国家“985工程”优势学科创新平台建设高校',
 '。',
 '90余年的办学历史形成了自己独特的办学特色、文化传统、精神内涵和社会声誉',
 '。',
 '\n我们的西财是最可爱的校园',
 '!',
 '您认为呢',
 '？',
 '\n']

In [12]:
print("\？")

\？


In [14]:
# remove "\n" in each sentence
sens = [s.replace("\n","") for s in sens]
sens

['西南财经大学是教育部直属的国家“211工程”和“985工程”优势学科创新平台建设的全国重点大学，也是国家“双一流”建设高校',
 '。',
 '学校地处国家历史文化名城、南方丝绸之路的起点、素有“天府之国”美誉的成都，有光华、柳林两校区，总占地2300余亩',
 '。',
 '光华铁树穿越年轮时光感受历史缅怀先贤,巍巍钟楼傲然屹立感受青春与未来对话,这里古今融通、传统与现代交相辉映，实乃读书治学的理想园地',
 '。',
 '学校始于1925年在上海创立的光华大学',
 '。',
 '1938年，因抗战西迁建立光华大学成都分部；1952-1953年汇聚西南地区17所院校的财经系科组建四川财经学院,这是建国初期国家按大区布局的四所本科财经院校之一，也是西南地区唯一的综合性高等财经学府;1960年后历经分设、合并、更名等，于1978年恢复为四川财经学院；1979年由四川省划归中国人民银行主管，逐渐形成了学校独特的金融行业背景和出色的金融学科优势；1985年更名为西南财经大学，1997年成为国家“211工程”重点建设高校，2000年以独立建制划转教育部管理，2011年成为国家“985工程”优势学科创新平台建设高校',
 '。',
 '90余年的办学历史形成了自己独特的办学特色、文化传统、精神内涵和社会声誉',
 '。',
 '我们的西财是最可爱的校园',
 '!',
 '您认为呢',
 '？',
 '']

**How to combine the above sentences and punctuation into sentences?**

In [15]:
new_sents = []
for i in range(int(len(sens)/2)):
    sent = sens[2*i] + sens[2*i+1] ## 偶数与奇数成对匹配
    new_sents.append(sent)
new_sents

['西南财经大学是教育部直属的国家“211工程”和“985工程”优势学科创新平台建设的全国重点大学，也是国家“双一流”建设高校。',
 '学校地处国家历史文化名城、南方丝绸之路的起点、素有“天府之国”美誉的成都，有光华、柳林两校区，总占地2300余亩。',
 '光华铁树穿越年轮时光感受历史缅怀先贤,巍巍钟楼傲然屹立感受青春与未来对话,这里古今融通、传统与现代交相辉映，实乃读书治学的理想园地。',
 '学校始于1925年在上海创立的光华大学。',
 '1938年，因抗战西迁建立光华大学成都分部；1952-1953年汇聚西南地区17所院校的财经系科组建四川财经学院,这是建国初期国家按大区布局的四所本科财经院校之一，也是西南地区唯一的综合性高等财经学府;1960年后历经分设、合并、更名等，于1978年恢复为四川财经学院；1979年由四川省划归中国人民银行主管，逐渐形成了学校独特的金融行业背景和出色的金融学科优势；1985年更名为西南财经大学，1997年成为国家“211工程”重点建设高校，2000年以独立建制划转教育部管理，2011年成为国家“985工程”优势学科创新平台建设高校。',
 '90余年的办学历史形成了自己独特的办学特色、文化传统、精神内涵和社会声誉。',
 '我们的西财是最可爱的校园!',
 '您认为呢？']

# open

## open file and  tokenize Chinese

#### 要用**open()**函数打开一个文件，就要向它传递一个字符串路径，表明希望打开的文件。这既可以是绝对路径，也可以是相对路径。open()函数返回一个File对象。

<center>**<变量名> = open(<文件名>, <打开模式>)**</center>

|打开模式|含义|
|-----|:-----|
|'r'|只读模式，如果文件不存在，返回异常FileNotFoundError，默认值|
|'w'|覆盖写模式，文件不存在则创建，存在则完全覆盖源文件|
|'x'|创建写模式，文件不存在则创建，存在则返回异常FileExistsError|
|'a'|追加写模式，文件不存在则创建，存在则在原文件最后追加内容|
|'b'|二进制文件模式|
|'t'|文本文件模式，默认值|
|'+'|与r/w/x/a一同使用，在原功能基础上增加同时读写功能|

In [17]:
with open("news.txt","r",encoding = "utf-8") as f: # with open 自动完成内存清理工作;open的结果赋值给f
    news = f.read()

In [18]:
news

'这是弘扬多边主义、彰显国际合作力量的时刻。凝聚多边之力，共同维护各国人民的健康和安全，正成为越来越多国家和地区的积极选择\n\u3000\u3000突如其来的新冠肺炎疫情提醒世界，这是一个传统安全与非传统安全相互交织的时代，也是一个局部问题和全球问题彼此转化的时代，人类生存依赖性日益紧密，人类命运息息相关。2月14日至16日第五十六届慕尼黑安全会议对全球公共卫生安全和新冠肺炎疫情的关注，再次说明了这一点。\n\u3000\u3000中方在会上介绍中国上下一心抗击疫情的行动和成效，得到与会者普遍称赞；世界卫生组织总干事谭德塞专程参会，肯定中国采取的从源头上控制疫情的措施令人鼓舞，并再次呼吁国际社会团结起来。在抗击疫情这场没有硝烟的战争中，世界不分西东，无论南北，是一个命运共同体。\n\u3000\u3000中国果断采取最全面、最严格、最彻底的举措迎击疫情，就是对全球防控疫情作出的最大贡献。正是因为中国速度、中国效率发挥的重要防控疫情效应，正是因为中国积极开展国际合作，中国境外确诊病例占所有病例不足1%。“我已经多次称赞中国，我还会继续这么做。”谭德塞的肺腑之言代表了国际社会的心声，众志成城冲锋抗疫一线的中国，完全配得上这样的评价和赞誉。\n\u3000\u3000世界支持中国，就是在为本国以及全球疫情防控贡献力量。在抗击疫情最艰苦的日子里，中国并不孤单，各国人民和中国人民坚定站在一起。全球160多个国家和国际组织的领导人专门发函致电中国，表达对中方的坚定支持。很多国家社会各界积极行动起来，或捐款捐物，或加油鼓劲。印尼警察演唱“武汉加油歌”，英国小学生合唱“让世界充满爱”，斯里兰卡各界为中国祈福……疫情面前，释放团结的力量，是各国人民发自内心作出的选择。\n\u3000\u3000这场疫情带给世界的启示是深刻的。慕尼黑安全会议会场内充斥各种各样的争论和交锋，一些与会者对所谓“西方缺失”的忧虑折射出安全感不足的心绪。人们的确应当思考，何为安全，如何安全？世界需要共同、综合、合作、可持续的安全。面对全球性挑战，没有哪个国家可以独善其身，也没有哪个国家可以包打天下。各国迫切需要摆脱区分东西方划界束缚，弥合凸显南北方差异的经济鸿沟，真正把人类赖以生存的星球看作一个生命共同体，把国际社会看作世界大家庭，共同构建人类命运共同体。\n\u3000\u3000如何应对全球公共卫生安全

In [19]:
import chardet 

# 获取文件编码类型

def get_encoding(file):
    # 二进制方式读取，获取字节数据，检测类型
    with open(file,"rb") as f:
        data = f.read()
        return chardet.detect(data)["encoding"]
    
file_name = "news.txt"
encoding = get_encoding(file_name)
print(encoding)

utf-8


In [18]:
news = news.replace("\n","") # 去掉换行符
news = news.replace("\u3000","") # 去掉空格
r = jieba.lcut(news)
#print(r)
"/".join(r)

'这是/弘扬/多边/主义/、/彰显/国际/合作/力量/的/时刻/。/凝聚/多边/之力/，/共同/维护/各国/人民/的/健康/和/安全/，/正/成为/越来越/多/国家/和/地区/的/积极/选择/突如其来/的/新冠/肺炎/疫情/提醒/世界/，/这是/一个/传统/安全/与/非传统/安全/相互交织/的/时代/，/也/是/一个/局部/问题/和/全球/问题/彼此/转化/的/时代/，/人类/生存/依赖性/日益/紧密/，/人类/命运/息息相关/。/2/月/14/日至/16/日/第五/十六届/慕尼黑/安全/会议/对/全球/公共卫生/安全/和/新冠/肺炎/疫情/的/关注/，/再次/说明/了/这/一点/。/中方/在/会上/介绍/中国/上下一心/抗击/疫情/的/行动/和/成效/，/得到/与会者/普遍/称赞/；/世界卫生组织/总干事/谭/德塞/专程/参会/，/肯定/中国/采取/的/从/源头/上/控制/疫情/的/措施/令人鼓舞/，/并/再次/呼吁/国际/社会/团结起来/。/在/抗击/疫情/这场/没有/硝烟/的/战争/中/，/世界/不/分/西东/，/无论/南北/，/是/一个/命运/共同体/。/中国/果断/采取/最/全面/、/最/严格/、/最/彻底/的/举措/迎击/疫情/，/就是/对/全球/防控/疫情/作出/的/最大/贡献/。/正是/因为/中国/速度/、/中国/效率/发挥/的/重要/防控/疫情/效应/，/正是/因为/中国/积极开展/国际/合作/，/中国/境外/确诊/病例/占/所有/病例/不足/1%/。/“/我/已经/多次/称赞/中国/，/我/还/会/继续/这么/做/。/”/谭/德塞/的/肺腑之言/代表/了/国际/社会/的/心声/，/众志成城/冲锋/抗疫/一线/的/中国/，/完全/配得/上/这样/的/评价/和/赞誉/。/世界/支持/中国/，/就是/在/为/本国/以及/全球/疫情/防控/贡献力量/。/在/抗击/疫情/最/艰苦/的/日子/里/，/中国/并/不/孤单/，/各国/人民/和/中国/人民/坚定/站/在/一起/。/全球/160/多个/国家/和/国际/组织/的/领导人/专门/发函/致电/中国/，/表达/对/中方/的/坚定/支持/。/很多/国家/社会各界/积极行动/起来/，/或/捐款捐物/，/或/加油/鼓劲/。/印尼/警察/演唱/“/武汉/加油/歌/”/，/英国/小学生/合唱/“/让/世界/充满/爱/

## read and readlines

The readlines() method is used to read all lines(up to the terminator EOF) and return the list, which can be supplied by Python's for... The in... The structure is treated.**文档进行多行读取，每一行作为一个元素，放在列表里面。**

The read() method is used to read the specified number of bytes from the file, or all if not given or negative.

In [19]:
with open("news.txt","r",encoding = "utf-8") as f:
    r = f.read()
r 

'这是弘扬多边主义、彰显国际合作力量的时刻。凝聚多边之力，共同维护各国人民的健康和安全，正成为越来越多国家和地区的积极选择\n\u3000\u3000突如其来的新冠肺炎疫情提醒世界，这是一个传统安全与非传统安全相互交织的时代，也是一个局部问题和全球问题彼此转化的时代，人类生存依赖性日益紧密，人类命运息息相关。2月14日至16日第五十六届慕尼黑安全会议对全球公共卫生安全和新冠肺炎疫情的关注，再次说明了这一点。\n\u3000\u3000中方在会上介绍中国上下一心抗击疫情的行动和成效，得到与会者普遍称赞；世界卫生组织总干事谭德塞专程参会，肯定中国采取的从源头上控制疫情的措施令人鼓舞，并再次呼吁国际社会团结起来。在抗击疫情这场没有硝烟的战争中，世界不分西东，无论南北，是一个命运共同体。\n\u3000\u3000中国果断采取最全面、最严格、最彻底的举措迎击疫情，就是对全球防控疫情作出的最大贡献。正是因为中国速度、中国效率发挥的重要防控疫情效应，正是因为中国积极开展国际合作，中国境外确诊病例占所有病例不足1%。“我已经多次称赞中国，我还会继续这么做。”谭德塞的肺腑之言代表了国际社会的心声，众志成城冲锋抗疫一线的中国，完全配得上这样的评价和赞誉。\n\u3000\u3000世界支持中国，就是在为本国以及全球疫情防控贡献力量。在抗击疫情最艰苦的日子里，中国并不孤单，各国人民和中国人民坚定站在一起。全球160多个国家和国际组织的领导人专门发函致电中国，表达对中方的坚定支持。很多国家社会各界积极行动起来，或捐款捐物，或加油鼓劲。印尼警察演唱“武汉加油歌”，英国小学生合唱“让世界充满爱”，斯里兰卡各界为中国祈福……疫情面前，释放团结的力量，是各国人民发自内心作出的选择。\n\u3000\u3000这场疫情带给世界的启示是深刻的。慕尼黑安全会议会场内充斥各种各样的争论和交锋，一些与会者对所谓“西方缺失”的忧虑折射出安全感不足的心绪。人们的确应当思考，何为安全，如何安全？世界需要共同、综合、合作、可持续的安全。面对全球性挑战，没有哪个国家可以独善其身，也没有哪个国家可以包打天下。各国迫切需要摆脱区分东西方划界束缚，弥合凸显南北方差异的经济鸿沟，真正把人类赖以生存的星球看作一个生命共同体，把国际社会看作世界大家庭，共同构建人类命运共同体。\n\u3000\u3000如何应对全球公共卫生安全

In [20]:
with open("news.txt","r",encoding = "utf-8") as f:
    r = f.readlines() # 一次性读取出来
r # r是一个列表

['这是弘扬多边主义、彰显国际合作力量的时刻。凝聚多边之力，共同维护各国人民的健康和安全，正成为越来越多国家和地区的积极选择\n',
 '\u3000\u3000突如其来的新冠肺炎疫情提醒世界，这是一个传统安全与非传统安全相互交织的时代，也是一个局部问题和全球问题彼此转化的时代，人类生存依赖性日益紧密，人类命运息息相关。2月14日至16日第五十六届慕尼黑安全会议对全球公共卫生安全和新冠肺炎疫情的关注，再次说明了这一点。\n',
 '\u3000\u3000中方在会上介绍中国上下一心抗击疫情的行动和成效，得到与会者普遍称赞；世界卫生组织总干事谭德塞专程参会，肯定中国采取的从源头上控制疫情的措施令人鼓舞，并再次呼吁国际社会团结起来。在抗击疫情这场没有硝烟的战争中，世界不分西东，无论南北，是一个命运共同体。\n',
 '\u3000\u3000中国果断采取最全面、最严格、最彻底的举措迎击疫情，就是对全球防控疫情作出的最大贡献。正是因为中国速度、中国效率发挥的重要防控疫情效应，正是因为中国积极开展国际合作，中国境外确诊病例占所有病例不足1%。“我已经多次称赞中国，我还会继续这么做。”谭德塞的肺腑之言代表了国际社会的心声，众志成城冲锋抗疫一线的中国，完全配得上这样的评价和赞誉。\n',
 '\u3000\u3000世界支持中国，就是在为本国以及全球疫情防控贡献力量。在抗击疫情最艰苦的日子里，中国并不孤单，各国人民和中国人民坚定站在一起。全球160多个国家和国际组织的领导人专门发函致电中国，表达对中方的坚定支持。很多国家社会各界积极行动起来，或捐款捐物，或加油鼓劲。印尼警察演唱“武汉加油歌”，英国小学生合唱“让世界充满爱”，斯里兰卡各界为中国祈福……疫情面前，释放团结的力量，是各国人民发自内心作出的选择。\n',
 '\u3000\u3000这场疫情带给世界的启示是深刻的。慕尼黑安全会议会场内充斥各种各样的争论和交锋，一些与会者对所谓“西方缺失”的忧虑折射出安全感不足的心绪。人们的确应当思考，何为安全，如何安全？世界需要共同、综合、合作、可持续的安全。面对全球性挑战，没有哪个国家可以独善其身，也没有哪个国家可以包打天下。各国迫切需要摆脱区分东西方划界束缚，弥合凸显南北方差异的经济鸿沟，真正把人类赖以生存的星球看作一个生命共同体，把国际社会看作世界大家庭，共同构建人类命运共同体。

In [21]:
with open("news.txt","r",encoding = "utf-8") as f:
    r = f.readlines()
for line in r:
    print(line)

这是弘扬多边主义、彰显国际合作力量的时刻。凝聚多边之力，共同维护各国人民的健康和安全，正成为越来越多国家和地区的积极选择

　　突如其来的新冠肺炎疫情提醒世界，这是一个传统安全与非传统安全相互交织的时代，也是一个局部问题和全球问题彼此转化的时代，人类生存依赖性日益紧密，人类命运息息相关。2月14日至16日第五十六届慕尼黑安全会议对全球公共卫生安全和新冠肺炎疫情的关注，再次说明了这一点。

　　中方在会上介绍中国上下一心抗击疫情的行动和成效，得到与会者普遍称赞；世界卫生组织总干事谭德塞专程参会，肯定中国采取的从源头上控制疫情的措施令人鼓舞，并再次呼吁国际社会团结起来。在抗击疫情这场没有硝烟的战争中，世界不分西东，无论南北，是一个命运共同体。

　　中国果断采取最全面、最严格、最彻底的举措迎击疫情，就是对全球防控疫情作出的最大贡献。正是因为中国速度、中国效率发挥的重要防控疫情效应，正是因为中国积极开展国际合作，中国境外确诊病例占所有病例不足1%。“我已经多次称赞中国，我还会继续这么做。”谭德塞的肺腑之言代表了国际社会的心声，众志成城冲锋抗疫一线的中国，完全配得上这样的评价和赞誉。

　　世界支持中国，就是在为本国以及全球疫情防控贡献力量。在抗击疫情最艰苦的日子里，中国并不孤单，各国人民和中国人民坚定站在一起。全球160多个国家和国际组织的领导人专门发函致电中国，表达对中方的坚定支持。很多国家社会各界积极行动起来，或捐款捐物，或加油鼓劲。印尼警察演唱“武汉加油歌”，英国小学生合唱“让世界充满爱”，斯里兰卡各界为中国祈福……疫情面前，释放团结的力量，是各国人民发自内心作出的选择。

　　这场疫情带给世界的启示是深刻的。慕尼黑安全会议会场内充斥各种各样的争论和交锋，一些与会者对所谓“西方缺失”的忧虑折射出安全感不足的心绪。人们的确应当思考，何为安全，如何安全？世界需要共同、综合、合作、可持续的安全。面对全球性挑战，没有哪个国家可以独善其身，也没有哪个国家可以包打天下。各国迫切需要摆脱区分东西方划界束缚，弥合凸显南北方差异的经济鸿沟，真正把人类赖以生存的星球看作一个生命共同体，把国际社会看作世界大家庭，共同构建人类命运共同体。

　　如何应对全球公共卫生安全挑战？惟有同舟共济、共克时艰。这是弘扬多边主义、彰显国际合作力量的时刻——七十七国集团和联合国积极声援中国抗击新冠肺炎疫情

In [22]:
with open("news.txt","r",encoding = "utf-8") as f: 
    for line in f.readlines(): ## 如果文本比较大的情况，可以一行一行读取
        print (line)

这是弘扬多边主义、彰显国际合作力量的时刻。凝聚多边之力，共同维护各国人民的健康和安全，正成为越来越多国家和地区的积极选择

　　突如其来的新冠肺炎疫情提醒世界，这是一个传统安全与非传统安全相互交织的时代，也是一个局部问题和全球问题彼此转化的时代，人类生存依赖性日益紧密，人类命运息息相关。2月14日至16日第五十六届慕尼黑安全会议对全球公共卫生安全和新冠肺炎疫情的关注，再次说明了这一点。

　　中方在会上介绍中国上下一心抗击疫情的行动和成效，得到与会者普遍称赞；世界卫生组织总干事谭德塞专程参会，肯定中国采取的从源头上控制疫情的措施令人鼓舞，并再次呼吁国际社会团结起来。在抗击疫情这场没有硝烟的战争中，世界不分西东，无论南北，是一个命运共同体。

　　中国果断采取最全面、最严格、最彻底的举措迎击疫情，就是对全球防控疫情作出的最大贡献。正是因为中国速度、中国效率发挥的重要防控疫情效应，正是因为中国积极开展国际合作，中国境外确诊病例占所有病例不足1%。“我已经多次称赞中国，我还会继续这么做。”谭德塞的肺腑之言代表了国际社会的心声，众志成城冲锋抗疫一线的中国，完全配得上这样的评价和赞誉。

　　世界支持中国，就是在为本国以及全球疫情防控贡献力量。在抗击疫情最艰苦的日子里，中国并不孤单，各国人民和中国人民坚定站在一起。全球160多个国家和国际组织的领导人专门发函致电中国，表达对中方的坚定支持。很多国家社会各界积极行动起来，或捐款捐物，或加油鼓劲。印尼警察演唱“武汉加油歌”，英国小学生合唱“让世界充满爱”，斯里兰卡各界为中国祈福……疫情面前，释放团结的力量，是各国人民发自内心作出的选择。

　　这场疫情带给世界的启示是深刻的。慕尼黑安全会议会场内充斥各种各样的争论和交锋，一些与会者对所谓“西方缺失”的忧虑折射出安全感不足的心绪。人们的确应当思考，何为安全，如何安全？世界需要共同、综合、合作、可持续的安全。面对全球性挑战，没有哪个国家可以独善其身，也没有哪个国家可以包打天下。各国迫切需要摆脱区分东西方划界束缚，弥合凸显南北方差异的经济鸿沟，真正把人类赖以生存的星球看作一个生命共同体，把国际社会看作世界大家庭，共同构建人类命运共同体。

　　如何应对全球公共卫生安全挑战？惟有同舟共济、共克时艰。这是弘扬多边主义、彰显国际合作力量的时刻——七十七国集团和联合国积极声援中国抗击新冠肺炎疫情